Ok so we are doing a binary classification model, and we can do logisitic regression, naive bayes, decision tree, random forest, or XGBoost models.

# Step 0: Import Packages

In [31]:
# Operational Packages
import numpy as np # Numerical calculations
import pandas as pd # Data handling

# Visualization Packages
import matplotlib.pyplot as plt # Lower-level graphics
import seaborn as sns # High-level graphics

# General Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV, PredefinedSplit # Train test split for model building and Grid Search for tuning
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score # Model Performance metrics
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix # Confusion Matrix

### Machine Learning Models

# Modeling and evaluation for K-means
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score 
from sklearn.preprocessing import StandardScaler # Scaling data 

# Decision Tree
from sklearn.tree import DecisionTreeClassifier # Decision Tree
# This function displays the splits of the tree
from sklearn.tree import plot_tree 

# Random Forest 
from sklearn.ensemble import RandomForestClassifier # Random Forest Classifier

# XGBoost 
from xgboost import XGBClassifier
# This is the function that helps plot feature importance 
from xgboost import plot_importance


# For saving Machine Learning Models once we fit them
import pickle

# Step 1: Exploratory Data Analysis (EDA)

In [32]:
# Load the dataset into a pandas dataframe
df = pd.read_csv('train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [33]:
#examine the shape of the dataset
df.shape

(891, 12)

The dataset contains 891 rows and 12 columns, so 891 passengers with 12 chararcteristics describing each one of them.

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [35]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


some initial concerns about the data is that is that there are over 100 entries missing their age and around 700 entries are missing a cabin number, I will have to investigate that.

I will now examine the missing values

In [48]:
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

There are 
* 177 missing entries for age
* 687 missing entries for cabin
* 2 missing entries for Embarked

In [37]:
mask = df["Age"].isna()
age_missing = df[mask]
age_missing

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
17,18,1,2,"Williams, Mr. Charles Eugene",male,NaN,0,0,244373,13.0000,NaN,S
19,20,1,3,"Masselmani, Mrs. Fatima",female,NaN,0,0,2649,7.2250,NaN,C
26,27,0,3,"Emir, Mr. Farred Chehab",male,NaN,0,0,2631,7.2250,NaN,C
28,29,1,3,"O'Dwyer, Miss. Ellen ""Nellie""",female,NaN,0,0,330959,7.8792,NaN,Q
...,...,...,...,...,...,...,...,...,...,...,...,...
859,860,0,3,"Razi, Mr. Raihed",male,NaN,0,0,2629,7.2292,NaN,C
863,864,0,3,"Sage, Miss. Dorothy Edith ""Dolly""",female,NaN,8,2,CA. 2343,69.5500,NaN,S
868,869,0,3,"van Melkebeke, Mr. Philemon",male,NaN,0,0,345777,9.5000,NaN,S
878,879,0,3,"Laleff, Mr. Kristo",male,NaN,0,0,349217,7.8958,NaN,S


I wonder if you don't have your age not listed, if there is a correlation with not having your cabin listed as well?

In [38]:
age_missing.info()

<class 'pandas.core.frame.DataFrame'>
Index: 177 entries, 5 to 888
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  177 non-null    int64  
 1   Survived     177 non-null    int64  
 2   Pclass       177 non-null    int64  
 3   Name         177 non-null    object 
 4   Sex          177 non-null    object 
 5   Age          0 non-null      float64
 6   SibSp        177 non-null    int64  
 7   Parch        177 non-null    int64  
 8   Ticket       177 non-null    object 
 9   Fare         177 non-null    float64
 10  Cabin        19 non-null     object 
 11  Embarked     177 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 18.0+ KB


There seems to be a very high correlation between missing cabin and missing age as of the 177 entries that are missing age 158 of them are also missing a cabin number

In [39]:
177 - 19

158

To handle the missing values for age, I will assign the missing values to the mean age of the dataset rounded to the nearest whole number

In [47]:
# assign the missing values to the mean age of the dataset rounded to the nearest whole number
df["Age"] = df["Age"].fillna(round(df["Age"].mean()))

#Check my work
df["Age"].isna().sum()

df.head(20)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,30.0,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


I will now examine the entries of the dataset that do have an entry for Cabin

In [45]:
mask = df["Cabin"].notna()
cabins = df[mask]
cabins.shape

(204, 12)

In [46]:
#Check to see of those that have their cabin listed how many survived
cabins["Survived"].sum()

136

It seems as though if you had your cabin listed, there was a better chance that you would survive

There seems to be way too many missing values, particularly in the Cabin column, to drop them. I will create a new column called cabin_listed that is 1 if you had a cabin listed and 0 if you did not have a cabin. I will create this column in the Feature Engineering section

I will now examine embarked missing values

In [49]:
mask = df["Embarked"].isna()
embarked_missing = df[mask]
embarked_missing

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
61,62,1,1,"Icard, Miss. Amelie",female,38.0,0,0,113572,80.0,B28,NaN
829,830,1,1,"Stone, Mrs. George Nelson (Martha Evelyn)",female,62.0,0,0,113572,80.0,B28,NaN


I will the missing values for embarked to the most common value

In [ ]:
# assign the missing values to the most common embarked
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

#Check my work
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
dtype: int64

Check for duplicated values

In [52]:
df.duplicated().sum()

0

Check for Class balance for our target variable Survived

In [53]:
df["Survived"].value_counts(normalize=True)

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

62% of the people in this set died and 38% of the people survived. There is a slight imbalance, but there is no need to correct for this imbalance

# Step 2: Feature Engineering

Create the cabin_listed mentioned in column from earlier 

In [54]:
# Create a dummy variable in the dataframe for the cabin column
df["cabin_listed"] = df["Cabin"].notna().astype(int)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,cabin_listed
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,1
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,0


Maybe information can be derived from which cabins people stayed in. It seems as though there were 1st, 2nd, and 3rd class cabins.

I can group each entry by first, second, and third class

In [56]:
cabins["Cabin"].value_counts()

Cabin
B96 B98        4
G6             4
C23 C25 C27    4
C22 C26        3
F33            3
              ..
E34            1
C7             1
C54            1
E36            1
C148           1
Name: count, Length: 147, dtype: int64

# Step 3: Split the data

# Step 4: Model Building

# Step 5: Model Evaluation